# Bundestagswahl Simulation mit echten Wahldaten

Dieses Notebook demonstriert am Beispiel der Bundestagswahl 2021, wie echte Wahldaten in die Wahlsimulation injiziert und davon ausgehend verschiedene mögliche Wahlverläufe simuliert werden können.

## 1. Konfiguration erstellen und Wahldaten für die erste Iteration einlesen

In [ ]:
import numpy as np

from ipres import SuperMajorityMargin, SeatDistributionMethod, MarginUnit
from ipres import ElectionEvaluator, contestantsFromParties, Election, ElectionConfig, ConstituenciesConfig, VoteMatrix
from ipres.bundestagswahl_loader import load_bundestagswahl_data
from ipres.election_config import QuotaCorrectionStrategy, ConstituencyRepresentation, DrawLotsStrategy

# Lade BTW 2021 Daten
constituencies_df, vote_matrix, party_names = load_bundestagswahl_data(2021)
constituencies_config = ConstituenciesConfig(constituencies=constituencies_df)
contestants = contestantsFromParties(party_names)

# ElectionConfig mit Seed für Reproduzierbarkeit
election_config = ElectionConfig(
    constituencies_config=constituencies_config,
    participating_parties=party_names,
    parliament_majority_margin = SuperMajorityMargin(10, MarginUnit.SEATS),
    draw_lots_strategy = DrawLotsStrategy.RANDOM,
    seed=8685
    # seed = 8685 Union gewinnt durch Losentscheid bei erforderlichen 10 Sitzen Regierungsvorteil und explizitem Auslosen (parliament_majority_margin = SuperMajorityMargin(10, MarginUnit.SEATS), draw_lots_strategy = DrawLotsStrategy.RANDOM)
    # seed=8685 FDP gewinnt bei erforderlichen 10 Sitzen Regierungsvorteil durch Verzicht auf qualifizierte Mehrheit (parliament_majority_margin = SuperMajorityMargin(10, MarginUnit.SEATS), draw_lots_strategy = DrawLotsStrategy.MARGINAL_LEAD)
    #seed=1169 #Union gewinnt durch Losentscheid bei 5% Minimalunterschied (SuperMajorityMargin(5, MarginUnit.PERCENT)
    #seed=6 #Union gewinnt direkt in der 5. Iteration bei 5% Minimalunterschied (SuperMajorityMargin(5, MarginUnit.PERCENT)
)
# Wahlgang aus echten Wahldaten erstellen (vote_matrix injection)
vote_matrix_real = VoteMatrix.generate(
    constituencies_config=constituencies_config,
    contestants=contestants,
    vote_matrix=vote_matrix  # Echte Daten injizieren!
)

print(f"Anzahl Wahlkreise: {len(constituencies_config.getConstituencyNames())}")
print(f"Anzahl Parteien: {len(contestants)}")
print(f"Wahlkreisrepräsentation: {'Ganzes Parlament' if election_config.constituency_representation == ConstituencyRepresentation.ENTIRE_PARLIAMENT else 'Regierungsfraktion'}")
print(f"Parlamentssitze: {election_config.parliamentarySeats}")
print(f"Regierungssitze: {election_config.getParliamentMajoritySeats()}")
print(f"Regierungsmehrheit: {round(election_config.getParliamentMajorityPercent(),2)} %")
print("Wahlgang aus echten Daten der Bundestagswahl 2021.")
print(f"\nGesamtstimmen pro Partei im ersten Wahlgang:")
print(vote_matrix_real.getVotes().sum(axis=0).sort_values(ascending=False))

## 2. Simulationslauf

In [ ]:
# Election mit echten Daten als erste Iteration
from ipres import ElectionRoundInput, ElectionRound, DrawLotsStrategy
from matplotlib import pyplot as plt
election = Election(electionConfig=election_config)

# ElectionRoundInput mit echten Wahldaten erstellen
iteration_input = ElectionRoundInput(
    election=election,
    constituencies_config=constituencies_config,
    contestants={c.name: c for c in contestants},
    ballot_majority_percent=election_config.getParliamentMajorityPercent(),  # Hier wird derselbe Schwellwert wie für die Parlamentsmehrheit verwendet; ein abweichender Wahlgang-Schwellwert wäre über election_config.getBallotMajorityPercent() möglich. (Dann würden aber andere Ergebnisse herauskommen.)
    draw_lots_strategy=election_config.draw_lots_strategy,
    rng = np.random.default_rng(election_config.seed),
    #rng=np.random.default_rng(1169), #Union gewinnt durch Losentscheid bei 5% Minimalunterschied
    #rng=np.random.default_rng(6), #Union gewinnt direkt in der 5. Iteration
    vote_matrix=vote_matrix_real  # Echte BTW 2021 Daten injizieren
)

# Erste Iteration mit echten Daten starten
iteration = election.start(iteration_input)
iteration.formCoalition("Union", ["CDU", "CSU"])
title = f"Iteration {iteration.getRoundNumber()} (echte Daten von BW 2021)"
fig = iteration.plot_vote_share_pie(title=title, min_percent = 2.0 )
display(fig)
plt.close(fig)

while iteration.hasNext():
    iterationInput = iteration.getNextRoundInput()
    iteration = ElectionRound.run(iterationInput)

    if not iteration.wasDecidedByLot():
        title = f"Iteration {iteration.getRoundNumber()}  (simulierter Wahlgang)"
        fig = iteration.plot_vote_share_pie(title=title)
        display(fig)
        plt.close(fig)

if iteration.wasDecidedByLot():
    print(f"Die Wahl wurde in der {iteration.getRoundNumber()}. Iteration durch ein Zufallsverfahren entschieden.")
    if iteration.getDrawLotsStrategy() == DrawLotsStrategy.RANDOM:
        print(f"Als Zufallsverfahren kam ein Losverfahren zum Einsatz.")
    elif iteration.getDrawLotsStrategy() == DrawLotsStrategy.MARGINAL_LEAD:
        print(f"Minimale Ergebnisunterschiede, die als zufällig angesehen werden können, haben die Wahl entschieden.")
    else:
        raise ValueError(f"Unbekannter DrawLotsStrategy: {iteration.getDrawLotsStrategy()}")


print(f"Der Gewinner ist: {election.getWinner().name}")

---

## Abstimmungsmatrix und Wahlkreiswichtigkeit

Die relative Stimmmatrix zeigt den normierten Stimmenanteil jeder Partei pro Wahlkreis.
Die Wichtigkeitsmatrix bewertet, für welche Partei ein Wahlkreis besonders bedeutsam ist — Grundlage der Wahlkreiszuordnung.

In [ ]:
from ipres import VoteMatrixAnalyzer
import pandas as pd

pd.set_option('display.max_rows', 10)
vm_analyzer = VoteMatrixAnalyzer(election.getFirstIteration().vote_matrix.getVotes())
display(vm_analyzer.show_relative_vote_matrix(styler=False))
display(vm_analyzer.show_constituency_importance_matrix(styler=False, decimals=4))

## Sitzverteilung anzeigen

In [ ]:
import pandas as pd
from ipres.allocation import ConstituencyAllocationMethod
evaluator = ElectionEvaluator(
    seed=election_config.seed,
    seat_distribution_method=SeatDistributionMethod.SAINTE_LAGUE,
    constituency_allocation_method=ConstituencyAllocationMethod.OPTIMAL,
    quota_correction_strategy=QuotaCorrectionStrategy.FAVOR_LARGE_PARTIES,
)
result = evaluator.evaluate(election)
seat_distribution_table = result.get_seat_distribution_table()
display(seat_distribution_table)

pie = result.plot_seat_share_pie(min_seats_for_display=10)
display(pie)
plt.close(pie)


## 2. Wahlkreiszuordnungen anzeigen

In [ ]:
# Zusammenfassung der Wahlkreiszuordnungen
summary_table = result.get_constituency_summary_table()
print("Zusammenfassung Wahlkreise nach Partei:")
display(summary_table)

display(result.get_constituency_assignment_table(sort_by="constituency"))
